In [1]:
pip install pandas geopy tqdm

   ---------------------------------------- 0.0/125.4 kB ? eta -:--:--
   --- ------------------------------------ 10.2/125.4 kB ? eta -:--:--
   --------- ----------------------------- 30.7/125.4 kB 435.7 kB/s eta 0:00:01
   ------------ -------------------------- 41.0/125.4 kB 330.3 kB/s eta 0:00:01
   ---------------------------- ---------- 92.2/125.4 kB 525.1 kB/s eta 0:00:01
   ---------------------------- ---------- 92.2/125.4 kB 525.1 kB/s eta 0:00:01
   -------------------------------------- 125.4/125.4 kB 492.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/40.7 kB ? eta -:--:--
   ---------------------------------------- 40.7/40.7 kB 1.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
# osm_match_schools_batch_fixed.py

import os
import re
import time
import pandas as pd
import requests
from difflib import SequenceMatcher
from tqdm import tqdm

# ===================== НАСТРОЙКИ =====================
IN_CSV  = "kindergartens_almaty_private.csv"
OUT_CSV = "kindergartens_almaty_private_osm_matched.csv"
OSM_DUMP_CSV = "almaty_osm_schools_dump.csv"
NOT_FOUND_CSV = "kindergartens_almaty_private_osm_not_found.csv"

# bbox Алматы
BBOX = (43.10, 76.70, 43.45, 77.15)

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

HEADERS = {"User-Agent": "almaty-osm-school-matcher/2.0"}

ALMATY_LAT = 43.2389
ALMATY_LON = 76.8897

# ===================== OSM =====================
def download_osm_schools(bbox):
    south, west, north, east = bbox

    q = f"""
    [out:json][timeout:180];
    (
        node["amenity"="kindergarten"]({south},{west},{north},{east});
        way["amenity"="kindergarten"]({south},{west},{north},{east});
        relation["amenity"="kindergarten"]({south},{west},{north},{east});

        // иногда размечены как building
        node["building"="kindergarten"]({south},{west},{north},{east});
        way["building"="kindergarten"]({south},{west},{north},{east});
        relation["building"="kindergarten"]({south},{west},{north},{east});
    );
    out center tags;
    """

    for base in OVERPASS_URLS:
        for attempt in range(5):
            try:
                r = requests.post(base, data=q.encode("utf-8"), headers=HEADERS, timeout=240)
                if r.status_code in (429, 502, 503, 504):
                    time.sleep(2 * (attempt + 1))
                    continue

                r.raise_for_status()
                js = r.json()

                rows = []
                for e in js.get("elements", []):
                    tags = e.get("tags", {}) or {}

                    if e["type"] == "node":
                        lat, lon = e.get("lat"), e.get("lon")
                    else:
                        c = e.get("center") or {}
                        lat, lon = c.get("lat"), c.get("lon")

                    if lat is None or lon is None:
                        continue

                    rows.append({
                        "osm_type": e["type"],
                        "osm_id": e["id"],
                        "name": tags.get("name"),
                        "name_ru": tags.get("name:ru"),
                        "lat": lat,
                        "lon": lon,
                    })

                return pd.DataFrame(rows)

            except Exception:
                time.sleep(2)

    raise RuntimeError("Overpass failed")

# ===================== MATCHING =====================
def norm(s):
    if s is None:
        return ""
    s = str(s).lower()
    s = s.replace("ё", "е").replace("№", " no ")
    s = re.sub(r"[^0-9a-zа-я\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def extract_number(s):
    s = norm(s)
    m = re.search(r"\bno\s*(\d+)", s)
    if m:
        return m.group(1)
    m = re.search(r"школ[а-я]*\s*(\d+)", s)
    if m:
        return m.group(1)
    return None

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def best_match(name, osm):
    target = norm(name)
    num = extract_number(name)

    cand = osm
    if num:
        tmp = osm[osm["name_norm"].str.contains(num, na=False)]
        if len(tmp) > 0:
            cand = tmp

    exact = cand[cand["name_norm"] == target]
    if len(exact) > 0:
        r = exact.iloc[0]
        return r, "exact", 1.0

    best = None
    best_score = 0

    for _, r in cand.iterrows():
        sc = similarity(target, r["name_norm"])
        if sc > best_score:
            best_score = sc
            best = r

    if best_score < (0.62 if num else 0.72):
        return None, None, None

    return best, "fuzzy", best_score

# ===================== GEOCODERS =====================
def in_bbox(lat, lon):
    return 43.10 <= lat <= 43.45 and 76.70 <= lon <= 77.15

photon_cache = {}
def photon_geocode(q):
    if q in photon_cache:
        return photon_cache[q]

    url = "https://photon.komoot.io/api"
    try:
        r = requests.get(url, params={"q": q, "limit": 1}, timeout=10)
        js = r.json()
        feats = js.get("features", [])
        if feats:
            lon, lat = feats[0]["geometry"]["coordinates"]
            if in_bbox(lat, lon):
                photon_cache[q] = (lat, lon)
                return lat, lon
    except:
        pass

    photon_cache[q] = (None, None)
    return None, None

# ===================== MAIN =====================
def main():
    df = pd.read_csv(IN_CSV)

    # OSM
    if os.path.exists(OSM_DUMP_CSV):
        osm = pd.read_csv(OSM_DUMP_CSV)
    else:
        osm = download_osm_schools(BBOX)
        osm.to_csv(OSM_DUMP_CSV, index=False)

    osm["name_best"] = osm["name_ru"].fillna(osm["name"]).fillna("")
    osm["name_norm"] = osm["name_best"].apply(norm)

    # OUTPUT
    out = df.copy()
    out["lat"] = None
    out["lon"] = None
    out["method"] = None
    out["score"] = None

    # 1. OSM matching
    for i, row in tqdm(out.iterrows(), total=len(out)):
        res, method, score = best_match(row["name"], osm)
        if res is not None:
            out.at[i, "lat"] = res["lat"]
            out.at[i, "lon"] = res["lon"]
            out.at[i, "method"] = method
            out.at[i, "score"] = score

    # 2. Photon fallback
    for i in tqdm(out[out["lat"].isna()].index):
        addr = str(out.at[i, "address"])
        lat, lon = photon_geocode(addr + ", Алматы")

        if lat:
            out.at[i, "lat"] = lat
            out.at[i, "lon"] = lon
            out.at[i, "method"] = "photon"

        time.sleep(0.2)

    # 3. Nominatim fallback
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter

    geo = Nominatim(user_agent="almaty-final")
    geocode = RateLimiter(geo.geocode, min_delay_seconds=1)

    for i in tqdm(out[out["lat"].isna()].index):
        addr = str(out.at[i, "address"])
        try:
            loc = geocode(addr + ", Алматы, Казахстан")
            if loc and in_bbox(loc.latitude, loc.longitude):
                out.at[i, "lat"] = loc.latitude
                out.at[i, "lon"] = loc.longitude
                out.at[i, "method"] = "nominatim"
        except:
            pass

    # confidence
    def conf(row):
        if row["method"] == "exact":
            return "high"
        if row["method"] == "fuzzy" and row["score"] > 0.8:
            return "high"
        if row["method"] in ["photon", "nominatim"]:
            return "medium"
        return "low"

    out["confidence"] = out.apply(conf, axis=1)

    out.to_csv(OUT_CSV, index=False)
    out[out["lat"].isna()].to_csv(NOT_FOUND_CSV, index=False)

    print("DONE")
    print("Found:", out["lat"].notna().sum())
    print("Not found:", out["lat"].isna().sum())

if __name__ == "__main__":
    main()

100%|██████████| 462/462 [07:49<00:00,  1.02s/it]

DONE
Found: 575
Not found: 155


In [3]:
import pandas as pd
import re
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

df = pd.read_csv("kindergartens_almaty_private_osm_matched.csv")

# viewbox Алматы
ALMATY_VIEWBOX = [(43.10, 76.70), (43.45, 77.15)]

geolocator = Nominatim(user_agent="almaty-clean-pass", timeout=20)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)


def clean_address(addr):
    if pd.isna(addr):
        return ""

    s = str(addr)
    s = s.replace("мкр.", "микрорайон")
    s = s.replace("ул.", "улица")
    s = s.replace("пр.", "проспект")
    s = re.sub(r"\s+", " ", s).strip(" ,")
    return s


def address_variants(addr):
    a = clean_address(addr)
    variants = [a]

    # без микрорайона
    v2 = re.sub(r"микрорайон[^,]+,\s*", "", a, flags=re.IGNORECASE)
    variants.append(v2)

    # только улица + дом
    m = re.search(r"(улица[^,]+,\s*\d+\w?)", a, flags=re.IGNORECASE)
    if m:
        variants.append(m.group(1))

    # убрать слова
    v4 = re.sub(r"(микрорайон|улица|проспект)", "", a, flags=re.IGNORECASE)
    variants.append(v4)

    # удалить пустые
    return list(dict.fromkeys([v.strip(" ,") for v in variants if v.strip()]))


def geo(q):
    try:
        loc = geocode(q, viewbox=ALMATY_VIEWBOX, bounded=True, country_codes="kz")
        if loc:
            return loc.latitude, loc.longitude
    except:
        pass
    return None, None


mask = df["lat"].isna()

for i in tqdm(df[mask].index):
    addr = df.loc[i, "address"]

    for v in address_variants(addr):
        lat, lon = geo(v + ", Алматы, Казахстан")
        if lat is not None:
            df.loc[i, "lat"] = lat
            df.loc[i, "lon"] = lon
            df.loc[i, "match_method"] = "clean_retry"
            break

df.to_csv("kindergartens_almaty_private_osm_matched_final.csv", index=False)

print("Найдено:", df["lat"].notna().sum())
print("Не найдено:", df["lat"].isna().sum())

100%|██████████| 155/155 [03:58<00:00,  1.54s/it]

Найдено: 676
Не найдено: 54


In [12]:
df['money']='yes'

In [2]:
import pandas as pd
df = pd.read_csv("kindergartens_almaty_private_osm_matched_final.csv")

In [25]:
df.to_csv("kindergartens_almaty_private_osm_matched_final.csv", index=False)

In [24]:
not_found = df[df["lat"].isna()].copy()
not_found

,name,address,phone,type,email,lat,lon,method,score,confidence,match_method,money


In [23]:
df.loc[df['name']=='Мини-центр "Медина-А"', 'lat'] = 43.174623
df.loc[df['name']=='Мини-центр "Медина-А"', 'lon'] = 76.795665

df.loc[df['name']=='Мини-центр "Медина-лидер"', 'lat'] = 43.20419
df.loc[df['name']=='Мини-центр "Медина-лидер"', 'lon'] = 76.802443

df.loc[df['name']=='Детский сад "Малика-2023"', 'lat'] = 43.22174
df.loc[df['name']=='Детский сад "Малика-2023"', 'lon'] = 76.791499

df.loc[df['name']=='Детский центр "Kinder land"', 'lat'] = 43.191871
df.loc[df['name']=='Детский центр "Kinder land"', 'lon'] = 76.815331

df.loc[df['name']=='Детский сад "Дарын2024"', 'lat'] = 43.191871
df.loc[df['name']=='Детский сад "Дарын2024"', 'lon'] = 76.815331

df.loc[df['name']=='Детский сад "Care Kids"', 'lat'] = 43.325604
df.loc[df['name']=='Детский сад "Care Kids"', 'lon'] = 76.981351
df.loc[df['name']=='Детский сад "Care Kids"', 'address'] = 'мкрн Жас Канат 550'


In [20]:
df[df['name']=='Детский сад "I-NAZ KIDS"']

,name,address,phone,type,email,lat,lon,method,score,confidence,match_method,money
224,"Детский сад ""I-NAZ KIDS""","ул.Шакшак Жанибека, 45/2","87018448740, 87273992587",детский сад,nurbekova_gaziza@mail.ru,43.273387,76.974304,NaN,NaN,low,NaN,yes


In [30]:
df[df.duplicated(subset=['lat', 'lon'])==True]

,name,address,phone,type,email,lat,lon,method,score,confidence,match_method,money
10,"Детский сад ""Аиша""","ул. Саялы, 14","87473002512, 87272514252",детский сад,akhat54@mail.ru,43.250226,76.899419,fuzzy,0.875000,high,NaN,yes
11,"Детский сад ""Мимишка""","ул. Димитрова, 10","87757526846, 87272570137",детский сад,mimishka_18@mail.ru,43.250226,76.899419,fuzzy,0.742857,low,NaN,yes
21,Детский сад «Гусельки»,"ул. Степана Разина, 94","87079614471, 87273854286",детский сад,guselki09@mail.ru,43.250226,76.899419,fuzzy,0.777778,low,NaN,yes
24,"Детский сад ""КҮН ШУАҚ""","ул. Дунентаева, 6/4","87016017606, 87016017606",детский сад,kun-shuak@mail.ru,43.250226,76.899419,fuzzy,0.742857,low,NaN,yes
30,"Детский сад ""АЗИМКА""","ул. Дегдара, 36/1","87071288851, 87071288851",детский сад,indigo2101@mail.ru,43.250226,76.899419,fuzzy,0.823529,high,NaN,yes
...,...,...,...,...,...,...,...,...,...,...,...,...
686,"ТОО ""АҚСАУЛЕ 2023""","мкр. Туркестан, 77","87074485570, 87074485570",детский сад,inkar_12@mail.ru,43.271812,76.852261,NaN,NaN,low,clean_retry,yes
693,"ТОО детский сад ""BB DAmir""","ул. Асанова, 7","87479582092, 87479582092",детский сад,damir.bb@bk.ru,43.337561,77.068094,NaN,NaN,low,clean_retry,yes
699,"ТОО ""Детский сад ""Мың бала Алматы""","ул. Казтуган Жырау, 29/1","87076602011, 87079335616",детский сад,mynbala2023@gmail.com,43.247665,76.813290,nominatim,NaN,medium,NaN,yes
705,"ТОО ""Детский сад Хан""","ул. Розыбакиева, 9","87075533537, 87478844851",детский сад,ail_220222@mail.ru,43.250226,76.899419,fuzzy,0.742857,low,NaN,yes


In [34]:
dups = df[df.duplicated(subset=['lat', 'lon'], keep=False)].sort_values(['lat', 'lon'])
print(dups[['name', 'address', 'lat', 'lon', 'method', 'score']].to_string())

                                                                  name                      address        lat        lon           method     score
403                                                 Детский сад "Нэмо"          мкр. Атамекен, 163А  43.207334  76.849053  nominatim_dedup  0.814815
405                               Детский сад "Білім балалар орталығы"          мкр. Атамекен, 163А  43.207334  76.849053              NaN       NaN
425                                   Детский-ясли сад "Береке - 2020"          мкр. Атамекен, 33/1  43.207593  76.844508  nominatim_dedup  0.723404
476                                        Детский сад "Тимас компани"          мкр. Атамекен, 33/1  43.207593  76.844508              NaN       NaN
387                                       Детский сад "Altyn Gasyr_KZ"             мкр. Мамыр-7, 21  43.212152  76.840487              NaN       NaN
440                                      Детский сад "Antaress kids-2"              мкр. Алмас, 106  43.21

In [32]:
import pandas as pd
import requests
import time

df = pd.read_csv("kindergartens_almaty_private_osm_matched_final.csv")

def in_bbox(lat, lon):
    return 43.10 <= lat <= 43.45 and 76.70 <= lon <= 77.15

def photon_geocode(q):
    try:
        r = requests.get("https://photon.komoot.io/api", 
                         params={"q": q, "limit": 3}, timeout=10)
        for feat in r.json().get("features", []):
            lon, lat = feat["geometry"]["coordinates"]
            if in_bbox(lat, lon):
                return lat, lon
    except:
        pass
    return None, None

# Найти дупликаты с разными адресами
dup_mask = df.duplicated(subset=['lat', 'lon'], keep=False)
dups = df[dup_mask].copy()

# Для каждой группы дупликатов — оставить первую запись, остальным сбросить координаты
rows_to_fix = []
for (lat, lon), group in dups.groupby(['lat', 'lon']):
    # Если адреса разные — все кроме первой нужно перегеокодировать
    unique_addresses = group['address'].nunique()
    if unique_addresses > 1:
        rows_to_fix.extend(group.index[1:].tolist())  # все кроме первой

print(f"Строк для перегеокодирования: {len(rows_to_fix)}")

# Перегеокодировать через Photon
for i in rows_to_fix:
    addr = df.loc[i, 'address']
    lat, lon = photon_geocode(f"{addr}, Алматы, Казахстан")
    if lat:
        df.loc[i, 'lat'] = lat
        df.loc[i, 'lon'] = lon
        df.loc[i, 'method'] = 'photon_dedup'
        print(f"✓ {df.loc[i, 'name']} → {lat:.5f}, {lon:.5f}")
    else:
        # Сбросить — лучше NaN чем неверные координаты
        df.loc[i, 'lat'] = None
        df.loc[i, 'lon'] = None
        df.loc[i, 'method'] = None
        print(f"✗ Не найдено: {df.loc[i, 'name']} | {addr}")
    time.sleep(0.3)

# Проверка результата
print(f"\nОсталось дупликатов: {df.duplicated(subset=['lat','lon']).sum()}")
print(f"Не найдено координат: {df['lat'].isna().sum()}")

df.to_csv("kindergartens_almaty_private_osm_matched_final.csv", index=False)

Строк для перегеокодирования: 273
✗ Не найдено: Детский сад "Tarlan kids" | ул. Кенесары хан, 67
✗ Не найдено: Детский сад "MARY POPPINS" | ул. Кали-Надырова, 98/1
✗ Не найдено: Филиал "Образовательного комплекса школы "First" | мкр. Мирас, 59
✗ Не найдено: Детский сад «Sparks Preschool» | мкр. Мирас, 160
✗ Не найдено: Детский центр "Kinder land" | ул. Ш. Айтматова, 71
✗ Не найдено: Детский сад "Р-Мурагер 2" | ул. Санаторная, 12
✗ Не найдено: Детский сад "Antaress kids-2" | мкр. Алмас, 106
✗ Не найдено: Мини-центр "ДиДи2016" (филиал) | мкр. Мамыр-7, 8/1
✗ Не найдено: Мини-центр "АйДиНур" | мкр. Мамыр 4, 156
✗ Не найдено: Образовательный центр "Арайлым" | мкр. Баянаул, 40
✗ Не найдено: Ақниет Образовательный Центр | мкр. Мамыр 4, 58
✗ Не найдено: Образовательный центр "Алладин" | ул. Кокшокы, 6а
✗ Не найдено: Детский сад "Munarat Almaty" | ул. Аширбек Сыгай, 96
✗ Не найдено: Детский сад «Бала- Дария» филиал | ул. Аширбек Сыгай, 55
✗ Не найдено: Детский сад «АкЕрке" | ул. Белбулак, 20
✗ 

In [33]:
import pandas as pd
import requests
import time
import re

df = pd.read_csv("kindergartens_almaty_private_osm_matched_final.csv")

def in_bbox(lat, lon):
    return 43.10 <= lat <= 43.45 and 76.70 <= lon <= 77.15

# --- Nominatim ---
def nominatim_geocode(q):
    try:
        r = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params={"q": q, "format": "json", "limit": 3,
                    "countrycodes": "kz", "addressdetails": 0},
            headers={"User-Agent": "almaty-dedup/1.0"},
            timeout=10
        )
        for res in r.json():
            lat, lon = float(res["lat"]), float(res["lon"])
            if in_bbox(lat, lon):
                return lat, lon
    except:
        pass
    return None, None

# --- 2GIS ---
def twogis_geocode(addr):
    try:
        r = requests.get(
            "https://geocode.maps.co/search",  # или можно попробовать другой
            params={"q": f"{addr}, Алматы", "format": "json"},
            timeout=10
        )
        for res in r.json():
            lat, lon = float(res["lat"]), float(res["lon"])
            if in_bbox(lat, lon):
                return lat, lon
    except:
        pass
    return None, None

# --- Очистка адреса ---
def clean_variants(addr):
    if pd.isna(addr):
        return []
    s = str(addr).strip()
    
    variants = []
    
    # Оригинал + город
    variants.append(f"{s}, Алматы, Казахстан")
    
    # Нормализация сокращений
    s2 = s.replace("мкр.", "микрорайон").replace("ул.", "улица") \
          .replace("пр.", "проспект").replace("пр-т", "проспект") \
          .replace("пер.", "переулок")
    variants.append(f"{s2}, Алматы")
    
    # Убрать микрорайон — оставить только улицу/дом
    s3 = re.sub(r"микрорайон\s+\S+,?\s*", "", s2, flags=re.IGNORECASE).strip(" ,")
    if s3 and s3 != s2:
        variants.append(f"{s3}, Алматы")
    
    # Только цифры из адреса (дом) + улица без дома
    m = re.match(r"(.+?),?\s*(\d+\w?)$", s2)
    if m:
        street = m.group(1).strip()
        variants.append(f"{street}, Алматы")
    
    return list(dict.fromkeys(v for v in variants if v.strip()))

# Строки с NaN координатами
to_fix = df[df["lat"].isna()].index.tolist()
print(f"Всего для геокодирования: {len(to_fix)}")

found = 0
for i in to_fix:
    addr = df.loc[i, "address"]
    name = df.loc[i, "name"]
    resolved = False
    
    for variant in clean_variants(addr):
        # 1. Nominatim
        lat, lon = nominatim_geocode(variant)
        if lat:
            df.loc[i, "lat"] = lat
            df.loc[i, "lon"] = lon
            df.loc[i, "method"] = "nominatim_dedup"
            print(f"✓ NOM  {name[:40]} | {variant[:50]}")
            resolved = True
            found += 1
            break
        time.sleep(1.1)  # Nominatim rate limit: 1 req/sec
    
    if not resolved:
        print(f"✗      {name[:40]} | {addr}")

print(f"\nНайдено: {found}/{len(to_fix)}")
print(f"Осталось NaN: {df['lat'].isna().sum()}")
print(f"Дупликатов: {df.duplicated(subset=['lat','lon']).sum()}")

df.to_csv("kindergartens_almaty_private_osm_matched_final.csv", index=False)

Всего для геокодирования: 273
✓ NOM  Детский сад "Аиша" | ул. Саялы, 14, Алматы, Казахстан
✓ NOM  Детский сад "Мимишка" | ул. Димитрова, 10, Алматы, Казахстан
✓ NOM  Детский сад «Гусельки» | ул. Степана Разина, 94, Алматы, Казахстан
✓ NOM  Детский сад "КҮН ШУАҚ" | ул. Дунентаева, 6/4, Алматы, Казахстан
✗      Детский сад "АЗИМКА" | ул. Дегдара, 36/1
✓ NOM  Детский сад "Тимошка" | улица Толегена Тажибаева, Алматы
✓ NOM  Детский сад "Жансая-Сәт" | ул. Зимняя, 18Б, Алматы, Казахстан
✓ NOM  Детский сад «Карусель» | ул. Стахановская, 19, Алматы, Казахстан
✓ NOM  Детский сад "Куанар" | улица Топчиева, Алматы
✓ NOM  Детский сад "Биржан К" | ул. Зимняя, 2, Алматы, Казахстан
✓ NOM  Детский сад "Адель&F" | проспект Суюнбая, 661, Алматы
✓ NOM  Детский сад "Кокетай" | ул. Дунентаева, 8/1, Алматы, Казахстан
✓ NOM  Детский сад «АкЕрке" | улица Белбулак, Алматы
✓ NOM  Детский сад "Балдаурен" | ул. Ижевская, 33, Алматы, Казахстан
✓ NOM  Детский сад "Кошахан" | 22-я улица, 46, Алматы, Казахстан
✓ NOM  

In [35]:
import pandas as pd
import requests
import time
import re

df = pd.read_csv("kindergartens_almaty_private_osm_matched_final.csv")

def in_bbox(lat, lon):
    return 43.10 <= lat <= 43.45 and 76.70 <= lon <= 77.15

# 2GIS неофициальный geocoder
def twogis_geocode(addr):
    try:
        r = requests.get(
            "https://catalog.api.2gis.com/3.0/items/geocode",
            params={
                "q": f"{addr}, Алматы",
                "fields": "items.point",
                "key": "rubimap",  # публичный демо-ключ
                "locale": "ru_KZ",
            },
            timeout=10
        )
        data = r.json()
        items = data.get("result", {}).get("items", [])
        for item in items:
            pt = item.get("point", {})
            lat, lon = pt.get("lat"), pt.get("lon")
            if lat and lon and in_bbox(lat, lon):
                return lat, lon
    except:
        pass
    return None, None

# Yandex geocoder (бесплатный лимит)
def yandex_geocode(addr):
    try:
        r = requests.get(
            "https://geocode-maps.yandex.ru/1.x/",
            params={
                "apikey": "демо",  # замените на свой ключ если есть
                "geocode": f"Алматы, {addr}",
                "format": "json",
                "ll": "76.89,43.24",
                "spn": "0.4,0.3",
                "rspn": 1,
                "results": 1,
            },
            timeout=10
        )
        coll = r.json()["response"]["GeoObjectCollection"]["featureMember"]
        if coll:
            pos = coll[0]["GeoObject"]["Point"]["pos"].split()
            lon, lat = float(pos[0]), float(pos[1])
            if in_bbox(lat, lon):
                return lat, lon
    except:
        pass
    return None, None

def clean_variants(addr):
    if pd.isna(addr):
        return []
    s = str(addr).strip()
    variants = [s]
    s2 = s.replace("мкр.", "микрорайон").replace("ул.", "улица") \
          .replace("пр.", "проспект").replace("пр-т", "проспект")
    variants.append(s2)
    # убрать микрорайон
    s3 = re.sub(r"микрорайон\s+\S+,?\s*", "", s2, flags=re.IGNORECASE).strip(" ,")
    if s3 and s3 != s2:
        variants.append(s3)
    return list(dict.fromkeys(v for v in variants if v.strip()))

to_fix = df[df["lat"].isna()].index.tolist()
print(f"Осталось геокодировать: {len(to_fix)}")

found = 0
still_missing = []

for i in to_fix:
    addr = df.loc[i, "address"]
    name = df.loc[i, "name"]
    resolved = False

    for variant in clean_variants(addr):
        # 1. 2GIS
        lat, lon = twogis_geocode(variant)
        if lat:
            df.loc[i, "lat"] = lat
            df.loc[i, "lon"] = lon
            df.loc[i, "method"] = "2gis"
            print(f"✓ 2GIS  {name[:40]} | {variant}")
            resolved = True
            found += 1
            break
        time.sleep(0.3)

    if not resolved:
        still_missing.append({"name": name, "address": addr})
        print(f"✗       {name[:40]} | {addr}")

print(f"\nНайдено: {found}/{len(to_fix)}")
print(f"Осталось NaN: {df['lat'].isna().sum()}")
print(f"Дупликатов: {df.duplicated(subset=['lat','lon']).sum()}")

df.to_csv("kindergartens_almaty_private_osm_matched_final.csv", index=False)

# Сохранить список которые не удалось найти вообще
if still_missing:
    pd.DataFrame(still_missing).to_csv("final_not_found.csv", index=False)
    print(f"Список не найденных → final_not_found.csv")

Осталось геокодировать: 129
✗       Детский сад "АЗИМКА" | ул. Дегдара, 36/1
✗       Детский центр "Kinder land" | ул. Ш. Айтматова, 71
✗       Детский сад "Top Kids 2" | ул. Кенжетаев, 119
✗       Детский сад "Top Kids" | ул. Кенжетаев, 121
✗       Детский сад "Тамирис" | ул. Кенесары-Хан, 116
✗       Детский сад "Талшын" | ул. Сейитбекова, 43
✗       Детский сад «Бала- Дария» филиал | ул. Аширбек Сыгай, 55
✗       Детский сад "Асылым" | ул. Ашимова, 259
✗       Детский сад "АЯНАТ" | ул. Аскарова, 28
✗       Детский сад "Дария Айша" | ул. Қыдырбеков, 60/1
✗       Детский сад "ALATAU" | ул. Афцинао, 11
✗       Детский сад "MARY POPPINS" | ул. Кали-Надырова, 98/1
✗       Детский сад "ЛИМПОПО" | ул. Аширбек Сыгай, 71
✗       Детский сад "БАТЫР-ШИФА" | ул. Кали Надырова, 4
✗       Детский сад "Балапан" | ул. Жалкамыс, 1
✗       Детский сад "Нұрбөпе" | ул. Герольда Бельгера, 57/1
✗       Детский сад "Ерсинтай" | ул. Аспандиярова, 83
✗       Детский сад "ЖАСДИМЕЙР" | ул. Кыдырова, 13
✗     

In [36]:
import pandas as pd
import requests
import time
import re

df = pd.read_csv("kindergartens_almaty_private_osm_matched_final.csv")

def in_bbox(lat, lon):
    return 43.10 <= lat <= 43.45 and 76.70 <= lon <= 77.15

def twogis_geocode(addr):
    try:
        r = requests.get(
            "https://catalog.api.2gis.com/3.0/items/geocode",
            params={
                "q": f"{addr}, Алматы",
                "fields": "items.point",
                "key": "rubimap",
                "locale": "ru_KZ",
            },
            timeout=10
        )
        for item in r.json().get("result", {}).get("items", []):
            pt = item.get("point", {})
            lat, lon = pt.get("lat"), pt.get("lon")
            if lat and lon and in_bbox(lat, lon):
                return lat, lon
    except Exception as e:
        print(f"  2GIS error: {e}")
    return None, None

to_fix = df[df["lat"].isna()].index.tolist()
print(f"Геокодируем: {len(to_fix)} садов")

found = 0
for i in to_fix:
    addr = str(df.loc[i, "address"])
    name = df.loc[i, "name"]
    
    # Пробуем оригинал и нормализованный адрес
    variants = [addr]
    addr2 = addr.replace("мкр.", "микрорайон").replace("ул.", "улица").replace("пр.", "проспект")
    if addr2 != addr:
        variants.append(addr2)
    
    resolved = False
    for v in variants:
        lat, lon = twogis_geocode(v)
        if lat:
            df.loc[i, "lat"] = lat
            df.loc[i, "lon"] = lon
            df.loc[i, "method"] = "2gis"
            print(f"✓ {name[:45]} | {v}")
            found += 1
            resolved = True
            break
        time.sleep(0.3)
    
    if not resolved:
        print(f"✗ {name[:45]} | {addr}")

print(f"\nНайдено: {found}/{len(to_fix)}")
print(f"Осталось NaN: {df['lat'].isna().sum()}")
print(f"Дупликатов: {df.duplicated(subset=['lat','lon']).sum()}")

df.to_csv("kindergartens_almaty_private_osm_matched_final11.csv", index=False)

Геокодируем: 129 садов
✗ Детский сад "АЗИМКА" | ул. Дегдара, 36/1
✗ Детский центр "Kinder land" | ул. Ш. Айтматова, 71
✗ Детский сад "Top Kids 2" | ул. Кенжетаев, 119
✗ Детский сад "Top Kids" | ул. Кенжетаев, 121
✗ Детский сад "Тамирис" | ул. Кенесары-Хан, 116
✗ Детский сад "Талшын" | ул. Сейитбекова, 43
✗ Детский сад «Бала- Дария» филиал | ул. Аширбек Сыгай, 55
✗ Детский сад "Асылым" | ул. Ашимова, 259
✗ Детский сад "АЯНАТ" | ул. Аскарова, 28
✗ Детский сад "Дария Айша" | ул. Қыдырбеков, 60/1
✗ Детский сад "ALATAU" | ул. Афцинао, 11
✗ Детский сад "MARY POPPINS" | ул. Кали-Надырова, 98/1
✗ Детский сад "ЛИМПОПО" | ул. Аширбек Сыгай, 71
✗ Детский сад "БАТЫР-ШИФА" | ул. Кали Надырова, 4
✗ Детский сад "Балапан" | ул. Жалкамыс, 1
✗ Детский сад "Нұрбөпе" | ул. Герольда Бельгера, 57/1
✗ Детский сад "Ерсинтай" | ул. Аспандиярова, 83
✗ Детский сад "ЖАСДИМЕЙР" | ул. Кыдырова, 13
✗ Детский сад "Кенгуренок-А" | ул. Сейтметова, 87
✗ Детский сад "Бала шак" | ул. Акбата, 68
✗ Детский сад "Сарқыт-А" | 

In [17]:
not_found.to_csv("not_found.csv", index=False)

In [3]:
# osm_match_schools_batch_fixed.py

import os
import re
import time
import pandas as pd
import requests
from difflib import SequenceMatcher
from tqdm import tqdm

# ===================== НАСТРОЙКИ =====================
IN_CSV  = "kindergartens_almaty_osm_matched.csv"
OUT_CSV = "kindergartens_almaty_osm_matched_final.csv"
OSM_DUMP_CSV = "kindergartens_almaty_osm_matched_dump.csv"
NOT_FOUND_CSV = "kindergartens_almaty_osm_not_found.csv"

# bbox Алматы
BBOX = (43.10, 76.70, 43.45, 77.15)

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

HEADERS = {"User-Agent": "almaty-osm-school-matcher/2.0"}

ALMATY_LAT = 43.2389
ALMATY_LON = 76.8897

# ===================== OSM =====================
def download_osm_schools(bbox):
    south, west, north, east = bbox

    q = f"""
    [out:json][timeout:180];
    (
        node["amenity"="kindergarten"]({south},{west},{north},{east});
        way["amenity"="kindergarten"]({south},{west},{north},{east});
        relation["amenity"="kindergarten"]({south},{west},{north},{east});

        // иногда размечены как building
        node["building"="kindergarten"]({south},{west},{north},{east});
        way["building"="kindergarten"]({south},{west},{north},{east});
        relation["building"="kindergarten"]({south},{west},{north},{east});
    );
    out center tags;
    """

    for base in OVERPASS_URLS:
        for attempt in range(5):
            try:
                r = requests.post(base, data=q.encode("utf-8"), headers=HEADERS, timeout=240)
                if r.status_code in (429, 502, 503, 504):
                    time.sleep(2 * (attempt + 1))
                    continue

                r.raise_for_status()
                js = r.json()

                rows = []
                for e in js.get("elements", []):
                    tags = e.get("tags", {}) or {}

                    if e["type"] == "node":
                        lat, lon = e.get("lat"), e.get("lon")
                    else:
                        c = e.get("center") or {}
                        lat, lon = c.get("lat"), c.get("lon")

                    if lat is None or lon is None:
                        continue

                    rows.append({
                        "osm_type": e["type"],
                        "osm_id": e["id"],
                        "name": tags.get("name"),
                        "name_ru": tags.get("name:ru"),
                        "lat": lat,
                        "lon": lon,
                    })

                return pd.DataFrame(rows)

            except Exception:
                time.sleep(2)

    raise RuntimeError("Overpass failed")

# ===================== MATCHING =====================
def norm(s):
    if s is None:
        return ""
    s = str(s).lower()
    s = s.replace("ё", "е").replace("№", " no ")
    s = re.sub(r"[^0-9a-zа-я\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def extract_number(s):
    s = norm(s)
    m = re.search(r"\bno\s*(\d+)", s)
    if m:
        return m.group(1)
    m = re.search(r"школ[а-я]*\s*(\d+)", s)
    if m:
        return m.group(1)
    return None

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def best_match(name, osm):
    target = norm(name)
    num = extract_number(name)

    cand = osm
    if num:
        tmp = osm[osm["name_norm"].str.contains(num, na=False)]
        if len(tmp) > 0:
            cand = tmp

    exact = cand[cand["name_norm"] == target]
    if len(exact) > 0:
        r = exact.iloc[0]
        return r, "exact", 1.0

    best = None
    best_score = 0

    for _, r in cand.iterrows():
        sc = similarity(target, r["name_norm"])
        if sc > best_score:
            best_score = sc
            best = r

    if best_score < (0.62 if num else 0.72):
        return None, None, None

    return best, "fuzzy", best_score

# ===================== GEOCODERS =====================
def in_bbox(lat, lon):
    return 43.10 <= lat <= 43.45 and 76.70 <= lon <= 77.15

photon_cache = {}
def photon_geocode(q):
    if q in photon_cache:
        return photon_cache[q]

    url = "https://photon.komoot.io/api"
    try:
        r = requests.get(url, params={"q": q, "limit": 1}, timeout=10)
        js = r.json()
        feats = js.get("features", [])
        if feats:
            lon, lat = feats[0]["geometry"]["coordinates"]
            if in_bbox(lat, lon):
                photon_cache[q] = (lat, lon)
                return lat, lon
    except:
        pass

    photon_cache[q] = (None, None)
    return None, None

# ===================== MAIN =====================
def main():
    df = pd.read_csv(IN_CSV)

    # OSM
    if os.path.exists(OSM_DUMP_CSV):
        osm = pd.read_csv(OSM_DUMP_CSV)
    else:
        osm = download_osm_schools(BBOX)
        osm.to_csv(OSM_DUMP_CSV, index=False)

    osm["name_best"] = osm["name_ru"].fillna(osm["name"]).fillna("")
    osm["name_norm"] = osm["name_best"].apply(norm)

    # OUTPUT
    out = df.copy()
    out["lat"] = None
    out["lon"] = None
    out["method"] = None
    out["score"] = None

    # 1. OSM matching
    for i, row in tqdm(out.iterrows(), total=len(out)):
        res, method, score = best_match(row["name"], osm)
        if res is not None:
            out.at[i, "lat"] = res["lat"]
            out.at[i, "lon"] = res["lon"]
            out.at[i, "method"] = method
            out.at[i, "score"] = score

    # 2. Photon fallback
    for i in tqdm(out[out["lat"].isna()].index):
        addr = str(out.at[i, "address"])
        lat, lon = photon_geocode(addr + ", Алматы")

        if lat:
            out.at[i, "lat"] = lat
            out.at[i, "lon"] = lon
            out.at[i, "method"] = "photon"

        time.sleep(0.2)

    # 3. Nominatim fallback
    from geopy.exc import GeocoderTimedOut, GeocoderUnavailable

    geo = Nominatim(user_agent="almaty-final", timeout=10)

    geocode = RateLimiter(
        geo.geocode,
        min_delay_seconds=1.2,
        max_retries=3,
        error_wait_seconds=5
    )
    
    for i in tqdm(out[out["lat"].isna()].index):
        addr = str(out.at[i, "address"])
        try:
            loc = geocode(addr + ", Алматы, Казахстан")
            if loc and in_bbox(loc.latitude, loc.longitude):
                out.at[i, "lat"] = loc.latitude
                out.at[i, "lon"] = loc.longitude
                out.at[i, "method"] = "nominatim"
        except (GeocoderTimedOut, GeocoderUnavailable):
            continue

    # confidence
    def conf(row):
        if row["method"] == "exact":
            return "high"
        if row["method"] == "fuzzy" and row["score"] > 0.8:
            return "high"
        if row["method"] in ["photon", "nominatim"]:
            return "medium"
        return "low"

    out["confidence"] = out.apply(conf, axis=1)

    out.to_csv(OUT_CSV, index=False)
    out[out["lat"].isna()].to_csv(NOT_FOUND_CSV, index=False)

    print("DONE")
    print("Found:", out["lat"].notna().sum())
    print("Not found:", out["lat"].isna().sum())

if __name__ == "__main__":
    main()

100%|██████████| 37/37 [00:43<00:00,  1.19s/it]

DONE
Found: 176
Not found: 22


In [3]:
import pandas as pd
import requests
import time
import re

df = pd.read_csv("kindergartens_almaty_private_osm_matched_final.csv")

YANDEX_KEY = "4e0aaf8d-4ab5-4be2-be63-7327bbe09df2"

def in_bbox(lat, lon):
    return 43.10 <= lat <= 43.45 and 76.70 <= lon <= 77.15

def clean(addr):
    """Убираем сокращения — геокодер лучше понимает полные слова или вообще без них"""
    addr = str(addr).strip()
    addr = re.sub(r'\bул\.\s*', '', addr)
    addr = re.sub(r'\bпр\.\s*', '', addr)
    addr = re.sub(r'\bпр-т\s*', '', addr)
    addr = re.sub(r'\bмкр\.\s*', 'микрорайон ', addr)
    addr = re.sub(r'\bпер\.\s*', '', addr)
    addr = re.sub(r'\s+', ' ', addr).strip(' ,')
    return addr

def yandex_geocode(addr):
    c = clean(addr)
    variants = [
        f"Алматы, {addr}",           # оригинал: Алматы, ул. Аскарова, 28
        f"Алматы, {c}",              # без сокращений: Алматы, Аскарова, 28
        f"Алматы {c}",               # без запятой: Алматы Аскарова 28
        f"Казахстан, Алматы, {c}",   # с полным путём
    ]
    
    for q in variants:
        try:
            r = requests.get(
                "https://geocode-maps.yandex.ru/1.x/",
                params={
                    "apikey": YANDEX_KEY,
                    "geocode": q,
                    "format": "json",
                    "ll": "76.89,43.24",
                    "spn": "0.35,0.25",
                    "rspn": 1,
                    "results": 1,
                    "lang": "ru_RU"
                },
                timeout=10
            )
            members = r.json()["response"]["GeoObjectCollection"]["featureMember"]
            if members:
                pos = members[0]["GeoObject"]["Point"]["pos"].split()
                lon, lat = float(pos[0]), float(pos[1])
                if in_bbox(lat, lon):
                    return lat, lon, q
        except Exception as e:
            print(f"  ошибка: {e}")
        time.sleep(0.2)
    
    return None, None, None

to_fix = df[df["lat"].isna()].index.tolist()
print(f"Геокодируем: {len(to_fix)} адресов\n")

found = 0
not_found = []

for i in to_fix:
    addr = str(df.loc[i, "address"])
    name = df.loc[i, "name"]
    
    lat, lon, matched_q = yandex_geocode(addr)
    
    if lat:
        df.loc[i, "lat"] = lat
        df.loc[i, "lon"] = lon
        df.loc[i, "method"] = "yandex"
        print(f"✓ {name[:45]}")
        print(f"  {addr} → {lat:.5f}, {lon:.5f}")
        found += 1
    else:
        not_found.append({"name": name, "address": addr})
        print(f"✗ {name[:45]} | {addr}")
    
    time.sleep(0.2)

print(f"\n{'='*50}")
print(f"Найдено:       {found}/{len(to_fix)}")
print(f"Осталось NaN:  {df['lat'].isna().sum()}")
print(f"Дупликатов:    {df.duplicated(subset=['lat','lon']).sum()}")

df.to_csv("kindergartens_almaty_private_osm_matched_final.csv", index=False)

if not_found:
    pd.DataFrame(not_found).to_csv("still_not_found.csv", index=False)
    print(f"\nНе найдено → still_not_found.csv ({len(not_found)} штук)")


Геокодируем: 9 адресов

✗ Детский сад "АЗИМКА" | ул. Дегдара, 36/1
✗ Детский сад «Бала- Дария» филиал | ул. Аширбек Сыгай, 55
✗ Детский сад "АЯНАТ" | ул. Аскарова, 28
✗ Детский сад "Мадина -27" | ул. Молодежная, 69А
✗ Детский сад «ЭЛКО» | мкр. Орбита 3, 8а
✗ Детский сад "Ахмадиева" | ул. Подгорная, 25/2
✗ Детский сад "Хан-Тәңірі-7" | ул. Василия Бенберина, 38/1
✗ Детский сад "Тұмар-А" | Квартал Жаңа құрылыс, 211
✗ Детский сад "Мәржол" | ул. Арқалық, 68

Найдено:       0/9
Осталось NaN:  9
Дупликатов:    44

Не найдено → still_not_found.csv (9 штук)


In [4]:
dups = df[df.duplicated(subset=["lat","lon"], keep=False)].sort_values(["lat","lon"])
print(dups[["name","address","lat","lon","method"]].to_string())


                                                                  name                      address        lat        lon           method
403                                                 Детский сад "Нэмо"          мкр. Атамекен, 163А  43.207334  76.849053  nominatim_dedup
405                               Детский сад "Білім балалар орталығы"          мкр. Атамекен, 163А  43.207334  76.849053              NaN
425                                   Детский-ясли сад "Береке - 2020"          мкр. Атамекен, 33/1  43.207593  76.844508  nominatim_dedup
476                                        Детский сад "Тимас компани"          мкр. Атамекен, 33/1  43.207593  76.844508              NaN
387                                       Детский сад "Altyn Gasyr_KZ"             мкр. Мамыр-7, 21  43.212152  76.840487              NaN
440                                      Детский сад "Antaress kids-2"              мкр. Алмас, 106  43.212152  76.840487  nominatim_dedup
451                        

In [52]:
import pandas as pd

# Для дупликатов с разными адресами — сбросить координаты у тех у кого method=NaN
# (они получили чужую точку)
dup_mask = df.duplicated(subset=["lat","lon"], keep=False)
dups = df[dup_mask].copy()

reset_count = 0
for (lat, lon), group in dups.groupby(["lat","lon"]):
    unique_addr = group["address"].nunique()
    if unique_addr > 1:
        # оставить только строки с method (они геокодировались сами)
        # сбросить строки у которых method=NaN — они "прилипли" к чужой точке
        nan_method_idx = group[group["method"].isna()].index
        df.loc[nan_method_idx, ["lat","lon"]] = None
        reset_count += len(nan_method_idx)

print(f"Сброшено ложных координат: {reset_count}")
print(f"Осталось NaN: {df['lat'].isna().sum()}")
print(f"Дупликатов: {df.duplicated(subset=['lat','lon']).sum()}")
print()
print("Строки без координат:")
print(df[df["lat"].isna()][["name","address"]].to_string())

df.to_csv("kindergartens_almaty_private_osm_matched_final11.csv", index=False)

Сброшено ложных координат: 0
Осталось NaN: 0
Дупликатов: 29

Строки без координат:
Empty DataFrame
Columns: [name, address]
Index: []


In [54]:
dups = df[df.duplicated(subset=["lat","lon"], keep=False)].sort_values(["lat","lon"])
print(f"Всего дупликатов: {len(dups)}")
print()
print(dups[["name","address","lat","lon","method"]].to_string())

Всего дупликатов: 55

                                                                  name                   address        lat        lon           method
403                                                 Детский сад "Нэмо"       мкр. Атамекен, 163А  43.207334  76.849053  nominatim_dedup
405                               Детский сад "Білім балалар орталығы"       мкр. Атамекен, 163А  43.207334  76.849053              NaN
425                                   Детский-ясли сад "Береке - 2020"       мкр. Атамекен, 33/1  43.207593  76.844508  nominatim_dedup
476                                        Детский сад "Тимас компани"       мкр. Атамекен, 33/1  43.207593  76.844508              NaN
440                                      Детский сад "Antaress kids-2"           мкр. Алмас, 106  43.212152  76.840487  nominatim_dedup
451                                               Детский сад "Айсана"           мкр. Алмас, 134  43.212152  76.840487  nominatim_dedup
473                       

In [52]:
dup_mask = df.duplicated(subset=["lat","lon"], keep=False)
dups = df[dup_mask].copy()

print("=== ОДИНАКОВЫЙ АДРЕС (два сада в одном здании) ===\n")
for (lat, lon), group in dups.groupby(["lat","lon"]):
    if group["address"].nunique() == 1:
        print(f"Адрес: {group['address'].iloc[0]}")
        for _, row in group.iterrows():
            print(f"  - {row['name']}")
        print()
        
        

=== ОДИНАКОВЫЙ АДРЕС (два сада в одном здании) ===



In [67]:
df = df[~df["name"].str.contains('ДиДи2016" (филиал)', na=False)]

C:\Users\Subbota\AppData\Local\Temp\ipykernel_25940\197973294.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["name"].str.contains('ДиДи2016" (филиал)', na=False)]


In [ ]:
df.loc[df['name']=='Детский сад "Еркем-ай"', 'address'] = 'ул. Парасат, 1'


In [79]:
df = df[df["name"] != 'Детский сад «Бала- Дария» филиал']


In [92]:
df.to_csv("kindergartens_almaty_private_osm_matched_final.csv", index=False)

In [84]:
df[df["name"].str.contains("Солнышко", na=False)]

,name,address,phone,type,email,lat,lon,method,score,confidence,match_method,money
193,"Мини центр ""Солнышко kz""","ул. Татибекова, 59/33","87758976758, 87273742575",дошкольный мини-центр,radik_avakriyev@mail.ru,43.2769,76.964097,nominatim,NaN,medium,NaN,yes


In [31]:
new_row = {
    "name": 'Детский сад "Бал-Дария-4"',
    "address": 'мкр. Достык, 128',
    "lat": 43.217869,
    "lon": 76.796721,
    'money': 'yes',
    'email': '',
    'phone': '',
    'type': 'частный детский сад'
}
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

In [ ]:
df.loc[df['name']=='Детский сад "Хан-Тәңірі-7"', 'lat'] = 43.274073
df.loc[df['name']=='Детский сад "Хан-Тәңірі-7"', 'lon'] = 76.845019

df.loc[df['name']=='Детский сад "Тұмар-А"', 'lat'] = 43.264514
df.loc[df['name']=='Детский сад "Тұмар-А"', 'lon'] = 76.790379

df.loc[df['name']=='Детский сад "Мәржол"', 'lat'] = 43.322593
df.loc[df['name']=='Детский сад "Мәржол"', 'lon'] = 76.841194

df.loc[df['name']=='ТОО "Балабакша "Болашақ"', 'lat'] = 43.336902
df.loc[df['name']=='ТОО "Балабакша "Болашақ"', 'lon'] = 76.844629


In [4]:
import pandas as pd
import re
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

df = pd.read_csv("kindergartens_almaty_osm_matched_final.csv")

# viewbox Алматы
ALMATY_VIEWBOX = [(43.45, 76.70), (43.10, 77.15)]

geolocator = Nominatim(user_agent="almaty-clean-pass", timeout=20)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)


def clean_address(addr):
    if pd.isna(addr):
        return ""

    s = str(addr)
    s = s.replace("мкр.", "микрорайон")
    s = s.replace("ул.", "улица")
    s = s.replace("пр.", "проспект")
    s = re.sub(r"\s+", " ", s).strip(" ,")
    return s


def address_variants(addr):
    a = clean_address(addr)
    variants = [a]

    # без микрорайона
    v2 = re.sub(r"микрорайон[^,]+,\s*", "", a, flags=re.IGNORECASE)
    variants.append(v2)

    # только улица + дом
    m = re.search(r"(улица[^,]+,\s*\d+\w?)", a, flags=re.IGNORECASE)
    if m:
        variants.append(m.group(1))

    # убрать слова
    v4 = re.sub(r"(микрорайон|улица|проспект)", "", a, flags=re.IGNORECASE)
    variants.append(v4)

    # удалить пустые
    return list(dict.fromkeys([v.strip(" ,") for v in variants if v.strip()]))
import requests

def in_bbox(lat, lon):
    return 43.10 <= lat <= 43.45 and 76.70 <= lon <= 77.15


photon_cache = {}

def photon_geocode(q):
    if q in photon_cache:
        return photon_cache[q]

    url = "https://photon.komoot.io/api"

    try:
        r = requests.get(url, params={"q": q, "limit": 1}, timeout=10)
        js = r.json()
        feats = js.get("features", [])

        if feats:
            lon, lat = feats[0]["geometry"]["coordinates"]

            if in_bbox(lat, lon):
                photon_cache[q] = (lat, lon)
                return lat, lon

    except:
        pass

    photon_cache[q] = (None, None)
    return None, None

def multi_geocode(addr):
    queries = [
        f"{addr}, Алматы, Казахстан",
        f"детский сад {addr}, Алматы",
        f"{addr}, Almaty, Kazakhstan"
    ]
    
    for q in queries:
        lat, lon = photon_geocode(q)
        if lat:
            return lat, lon
    
    return None, None
def geo(q):
    try:
        loc = geocode(q, viewbox=ALMATY_VIEWBOX, bounded=True, country_codes="kz")
        if loc:
            return loc.latitude, loc.longitude
    except:
        pass
    return None, None


mask = df["lat"].isna()

for i in tqdm(df[mask].index):
    addr = df.loc[i, "address"]
    for v in address_variants(addr):

        # 1. Photon
        lat, lon = multi_geocode(v)
        if lat is not None:
            df.loc[i, "lat"] = lat
            df.loc[i, "lon"] = lon
            df.loc[i, "match_method"] = "photon_clean"
            break

        # 2. Nominatim fallback
        lat, lon = geo(v)
        if lat is not None:
            df.loc[i, "lat"] = lat
            df.loc[i, "lon"] = lon
            df.loc[i, "match_method"] = "nominatim_clean"
            break

df.to_csv("kindergartens_almaty_osm_final.csv", index=False)

print("Найдено:", df["lat"].notna().sum())
print("Не найдено:", df["lat"].isna().sum())

100%|██████████| 22/22 [00:48<00:00,  2.22s/it]

Найдено: 197
Не найдено: 1


In [5]:
not_found = df[df["lat"].isna()].copy()
not_found

,name,address,phone,type,email,lat,lon,method,score,confidence,match_method
113,"КГКП ""Ясли-сад №87""","ул. Си-Синхая, 14","87015031313, 87273964336",ясли-сад,Sadik87@list.ru,NaN,NaN,NaN,NaN,low,NaN


In [91]:
import pandas as pd


dup_mask = df.duplicated(subset=["lat","lon"], keep=False)
dup_mask = dup_mask & df["lat"].notna()
dups = df[dup_mask].copy()

print("=== ОДИНАКОВЫЙ АДРЕС ===\n")
same_addr_count = 0
for (lat, lon), group in dups.groupby(["lat","lon"]):
    if group["address"].nunique() == 1:
        same_addr_count += 1
        print(f"Адрес: {group['address'].iloc[0]}")
        for _, row in group.iterrows():
            print(f"  - {row['name']}")
        print()

print(f"Всего групп с одинаковым адресом: {same_addr_count}")
print()

print("=== РАЗНЫЕ АДРЕСА (промах геокодера) ===\n")
diff_addr_count = 0
for (lat, lon), group in dups.groupby(["lat","lon"]):
    if group["address"].nunique() > 1:
        diff_addr_count += 1
        print(f"Точка: {lat}, {lon}")
        for _, row in group.iterrows():
            print(f"  - {row['name']} | {row['address']}")
        print()
        

print(f"Всего групп с разными адресами: {diff_addr_count}")


=== ОДИНАКОВЫЙ АДРЕС ===

Всего групп с одинаковым адресом: 0

=== РАЗНЫЕ АДРЕСА (промах геокодера) ===

Всего групп с разными адресами: 0


In [57]:
import pandas as pd
import requests
import time
import re

df = pd.read_csv("kindergartens_almaty_private_osm_matched_final.csv")

YANDEX_KEY = "4e0aaf8d-4ab5-4be2-be63-7327bbe09df2"

def in_bbox(lat, lon):
    return 43.10 <= lat <= 43.45 and 76.70 <= lon <= 77.15

def clean(addr):
    addr = str(addr).strip()
    addr = re.sub(r'\bул\.\s*', '', addr)
    addr = re.sub(r'\bпр\.\s*', '', addr)
    addr = re.sub(r'\bпр-т\s*', '', addr)
    addr = re.sub(r'\bмкр\.\s*', 'микрорайон ', addr)
    addr = re.sub(r'\bпер\.\s*', '', addr)
    addr = re.sub(r'\s+', ' ', addr).strip(' ,')
    return addr

def yandex_geocode(addr):
    c = clean(addr)
    variants = [
        f"Алматы, {addr}",
        f"Алматы, {c}",
        f"Алматы {c}",
        f"Казахстан, Алматы, {c}",
    ]
    for q in variants:
        try:
            r = requests.get(
                "https://geocode-maps.yandex.ru/1.x/",
                params={
                    "apikey": YANDEX_KEY,
                    "geocode": q,
                    "format": "json",
                    "ll": "76.89,43.24",
                    "spn": "0.35,0.25",
                    "rspn": 1,
                    "results": 1,
                    "lang": "ru_RU"
                },
                timeout=10
            )
            members = r.json()["response"]["GeoObjectCollection"]["featureMember"]
            if members:
                pos = members[0]["GeoObject"]["Point"]["pos"].split()
                lon, lat = float(pos[0]), float(pos[1])
                if in_bbox(lat, lon):
                    return lat, lon, q
        except Exception as e:
            print(f"  ошибка: {e}")
        time.sleep(0.2)
    return None, None, None

# ── 1. Геокодируем NaN ──────────────────────────────────────────────
to_fix = df[df["lat"].isna()].index.tolist()
print(f"Геокодируем NaN: {len(to_fix)} адресов\n")

found = 0
not_found = []

for i in to_fix:
    addr = str(df.loc[i, "address"])
    name = df.loc[i, "name"]
    lat, lon, matched_q = yandex_geocode(addr)
    if lat:
        df.loc[i, "lat"] = lat
        df.loc[i, "lon"] = lon
        df.loc[i, "method"] = "yandex"
        print(f"✓ {name[:45]}")
        print(f"  {addr} → {lat:.5f}, {lon:.5f}")
        found += 1
    else:
        not_found.append({"name": name, "address": addr})
        print(f"✗ {name[:45]} | {addr}")
    time.sleep(0.2)

print(f"\nNaN: найдено {found}/{len(to_fix)}")

# ── 2. Исправляем дубликаты координат (промахи геокодера) ───────────
print("\n" + "="*50)
print("Исправляем дубликаты координат...\n")

dup_mask = df.duplicated(subset=["lat", "lon"], keep=False) & df["lat"].notna()
dups = df[dup_mask].copy()

dup_fixed = 0
dup_not_found = []

for (lat, lon), group in dups.groupby(["lat", "lon"]):
    # Пропускаем группы с одинаковым адресом — там дубликат настоящий
    if group["address"].nunique() == 1:
        continue

    print(f"Точка {lat}, {lon} — {len(group)} садиков с разными адресами:")

    for i, row in group.iterrows():
        addr = str(row["address"])
        name = row["name"]

        new_lat, new_lon, matched_q = yandex_geocode(addr)

        if new_lat and (new_lat != lat or new_lon != lon):
            df.loc[i, "lat"] = new_lat
            df.loc[i, "lon"] = new_lon
            df.loc[i, "method"] = "yandex_dedup"
            print(f"  ✓ {name[:40]}")
            print(f"    {addr} → {new_lat:.5f}, {new_lon:.5f}")
            dup_fixed += 1
        else:
            # Геокодер снова вернул ту же точку или ничего — обнуляем
            df.loc[i, "lat"] = None
            df.loc[i, "lon"] = None
            df.loc[i, "method"] = "dedup_failed"
            dup_not_found.append({"name": name, "address": addr})
            print(f"  ✗ {name[:40]} | {addr} — обнулено")

        time.sleep(0.2)
    print()

# ── 3. Итог ─────────────────────────────────────────────────────────
print("="*50)
print(f"NaN геокодировано:       {found}/{len(to_fix)}")
print(f"Дубликатов исправлено:   {dup_fixed}")
print(f"Осталось NaN:            {df['lat'].isna().sum()}")
print(f"Осталось дубликатов:     {df.duplicated(subset=['lat','lon']).sum()}")

df.to_csv("kindergartens_almaty_private_osm_matched_final.csv", index=False)

all_not_found = not_found + dup_not_found
if all_not_found:
    pd.DataFrame(all_not_found).to_csv("still_not_found.csv", index=False)
    print(f"\nНе найдено → still_not_found.csv ({len(all_not_found)} штук)")

Геокодируем NaN: 6 адресов

✗ Детский сад "АЗИМКА" | ул. Дегдара, 36/1
✗ Детский сад «Бала- Дария» филиал | ул. Аширбек Сыгай, 55
✗ Детский сад "АЯНАТ" | ул. Аскарова, 28
✗ Детский сад "Мадина -27" | ул. Молодежная, 69А
✗ Детский сад «ЭЛКО» | мкр. Орбита 3, 8а
✗ Детский сад "Ахмадиева" | ул. Подгорная, 25/2

NaN: найдено 0/6

Исправляем дубликаты координат...

Точка 43.2121518, 76.8404866 — 5 садиков с разными адресами:
  ✓ Детский сад "Altyn Gasyr_KZ"
    мкр. Мамыр-7, 21 → 43.21009, 76.84090
  ✓ Детский сад "Antaress kids-2"
    мкр. Алмас, 106 → 43.20608, 76.83980
  ✓ Детский сад "Айсана"
    мкр. Алмас, 134 → 43.20437, 76.84146
  ✓ Детский сад "Ару kids"
    мкр. Алмас, 201 → 43.20685, 76.84215
  ✓ Детский сад "Садок ltd"
    мкр. Алмас, 172 → 43.20504, 76.84245

Точка 43.2147988, 76.8692698 — 2 садиков с разными адресами:
  ✓ Детский сад "Lion Garden"
    мкр. Алмас, 57а → 43.20770, 76.83878
  ✓ Детский сад "УМКА"
    мкр. Мамыр-4, 57А → 43.21631, 76.84888

Точка 43.2157126, 76.85

In [89]:
# Диагностика после прогона
print("=== Оставшиеся NaN ===")
print(df[df["lat"].isna()][["name", "address"]].to_string())

print("\n=== Оставшиеся дубликаты ===")
dup_mask = df.duplicated(subset=["lat","lon"], keep=False) & df["lat"].notna()
dups = df[dup_mask].copy()
for (lat, lon), group in dups.groupby(["lat","lon"]):
    print(f"\nТочка {lat:.5f}, {lon:.5f}:")
    for _, row in group.iterrows():
        print(f"  - {row['name']} | {row['address']} | method={row.get('method','')}")
        

=== Оставшиеся NaN ===
Empty DataFrame
Columns: [name, address]
Index: []

=== Оставшиеся дубликаты ===


In [88]:
df.loc[df['name']=='Детский сад "Арман"', 'lat'] = 43.337703
df.loc[df['name']=='Детский сад "Арман"', 'lon'] = 76.81359

In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 726 entries, 0 to 727
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          726 non-null    object 
 1   address       726 non-null    object 
 2   phone         720 non-null    object 
 3   type          724 non-null    object 
 4   email         719 non-null    object 
 5   lat           726 non-null    float64
 6   lon           726 non-null    float64
 7   method        577 non-null    object 
 8   score         265 non-null    float64
 9   confidence    721 non-null    object 
 10  match_method  98 non-null     object 
 11  money         726 non-null    object 
dtypes: float64(3), object(9)
memory usage: 73.7+ KB


In [ ]:
df.drop(columns=[""], inplace=True)

In [1]:
import osmnx as ox

north = 43.35
south = 43.15
east = 77.10
west = 76.70

G = ox.graph_from_place(
    "Almaty, Kazakhstan",
    network_type="walk"
)

ox.save_graphml(G, "almaty_walk_q01_q99.graphml")
